## 1. Load Dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("shortened_result-val.csv")
print("Original shape:", df.shape)
df.head()

## 2. Convert Visit Codes to Months (bl & sc both → 0)

In [ ]:
MAX_MONTH = 120
INVALID_VISITS = {"scmri", "AUT", "nv", "uns1"}

def visit_to_month(visit):
    if pd.isna(visit):
        return np.nan
    visit = str(visit).strip().lower()
    if visit in {v.lower() for v in INVALID_VISITS}:
        return np.nan
    if visit in ("bl", "sc"):   # both baseline and screening map to month 0
        return 0
    if visit.startswith("m"):
        try:
            return int(visit[1:])
        except ValueError:
            return np.nan
    return np.nan

df["visit_month"] = df["visit"].apply(visit_to_month)

# Drop invalid or out-of-range visits
df_clean = df[
    (df["visit_month"].notna()) &
    (df["visit_month"] <= MAX_MONTH)
].copy()

print("After dropping invalid / >120-month visits:", df_clean.shape)
df_clean["visit"].value_counts().head(10)

## 3. Merge bl & sc into a single m0 row per patient

For each subject, if *both* bl and sc exist at visit_month 0:
- Take the non-NaN value from either row (they carry the same value when both are present).
- Collapse them into one row labelled **m0**.

In [ ]:
TARGET_COLS = ["CDGLOBAL", "MMSCORE", "TOTSCORE"]
MERGE_COLS  = ["entry_age", "DIAGNOSIS"] + TARGET_COLS   # columns to coalesce

def merge_bl_sc(group):
    """
    Within a single subject's data, combine all rows at visit_month == 0
    (i.e. bl and sc) into one row.  Non-month-0 rows pass through unchanged.
    """
    m0_rows  = group[group["visit_month"] == 0]
    non_m0   = group[group["visit_month"] != 0].copy()

    if len(m0_rows) == 0:
        return non_m0                            # nothing at month 0

    # Start from the first row as the base, then fill in NaNs from the other
    merged = m0_rows.iloc[[0]].copy()
    for col in MERGE_COLS:
        # Take first non-NaN across all month-0 rows
        non_null = m0_rows[col].dropna()
        if len(non_null) > 0:
            merged[col] = non_null.iloc[0]

    merged["visit"]       = "m0"
    merged["visit_month"] = 0

    return pd.concat([merged, non_m0], ignore_index=True)

df_merged = (
    df_clean
    .groupby("subject_id", group_keys=False)
    .apply(merge_bl_sc)
    .sort_values(["subject_id", "visit_month"])
    .reset_index(drop=True)
)

print("Shape after merging bl/sc → m0:", df_merged.shape)
df_merged.head(10)

In [ ]:
# Verify: no bl or sc should remain
print("Remaining bl/sc rows:", df_merged["visit"].isin(["bl", "sc"]).sum())
print("m0 rows:", (df_merged["visit"] == "m0").sum())
print("\nVisit distribution (top 10):")
df_merged["visit"].value_counts().head(10)

## 4. Trajectory Profiling (per patient, per target column)

In [ ]:
def count_valid(series):
    return series.notna().sum()

def is_constant(series):
    vals = series.dropna().unique()
    return len(vals) == 1

def is_monotonic(series):
    vals = series.dropna().values
    if len(vals) < 3:
        return False
    return np.all(np.diff(vals) >= 0) or np.all(np.diff(vals) <= 0)

profiles = []
for pid, g in df_merged.groupby("subject_id"):
    profile = {"subject_id": pid}
    for col in TARGET_COLS:
        s = g[col]
        profile[f"{col}_valid"]     = count_valid(s)
        profile[f"{col}_constant"]  = is_constant(s)
        profile[f"{col}_monotonic"] = is_monotonic(s)
    profile["num_visits"] = len(g)
    profiles.append(profile)

traj_profile = pd.DataFrame(profiles)
traj_profile.head()

In [ ]:
def trajectory_type(row, col):
    if row[f"{col}_valid"] >= 3 and row[f"{col}_monotonic"]:
        return "monotonic"
    if row[f"{col}_valid"] >= 2 and row[f"{col}_constant"]:
        return "constant"
    if row[f"{col}_valid"] >= 2:
        return "sparse_trend"
    return "too_sparse"

for col in TARGET_COLS:
    traj_profile[f"{col}_traj_type"] = traj_profile.apply(
        lambda r, c=col: trajectory_type(r, c), axis=1
    )

for col in TARGET_COLS:
    print(f"\n{col} trajectory distribution (%)")
    print(traj_profile[f"{col}_traj_type"].value_counts(normalize=True) * 100)

## 5. Select Earliest 4 Visits per Patient

In [ ]:
def select_earliest_4(g):
    return g.sort_values("visit_month").head(4)

df_4 = (
    df_merged
    .groupby("subject_id", group_keys=False)
    .apply(select_earliest_4)
    .reset_index(drop=True)
)

# Attach trajectory types
df_4 = df_4.merge(
    traj_profile[
        ["subject_id"] + [f"{c}_traj_type" for c in TARGET_COLS]
    ],
    on="subject_id",
    how="left"
)

print("Shape after selecting earliest-4:", df_4.shape)
print("\nVisits per patient:")
df_4.groupby("subject_id").size().value_counts()

In [ ]:
# Missingness before imputation
print("Missing % before imputation:")
df_4[TARGET_COLS].isna().mean() * 100

## 6. Impute Missing Values (same strategy as original)

In [ ]:
def impute_column_patientwise(g, col, strategy):
    months = g["visit_month"]
    values = g[col].copy()

    if strategy == "constant":
        const_val = values.dropna().iloc[0]
        return values.fillna(const_val)

    if strategy == "monotonic":
        interp = (
            pd.Series(values.values, index=months)
            .interpolate(method="index", limit_area="inside")
        )
        return interp.ffill().bfill().values

    if strategy == "sparse_trend":
        return values.ffill().bfill()

    return values   # too_sparse → leave NaN


df_imputed = []

for pid, g in df_4.groupby("subject_id"):
    g = g.sort_values("visit_month").copy()
    for col in TARGET_COLS:
        traj_type = g[f"{col}_traj_type"].iloc[0]
        g[col] = impute_column_patientwise(g, col, traj_type)
    df_imputed.append(g)

df_imputed = pd.concat(df_imputed).reset_index(drop=True)

print("Missing % after imputation:")
df_imputed[TARGET_COLS].isna().mean() * 100

## 7. Imputation Confidence

In [ ]:
def imputation_confidence(row):
    if (
        row["CDGLOBAL_traj_type"] == "too_sparse" or
        row["MMSCORE_traj_type"]  == "too_sparse" or
        row["TOTSCORE_traj_type"] == "too_sparse"
    ):
        return "low"
    return "high"

df_imputed["imputation_confidence"] = df_imputed.apply(imputation_confidence, axis=1)
print("Confidence distribution (%):")
df_imputed["imputation_confidence"].value_counts(normalize=True) * 100

## 8. Save to CSV

In [ ]:
df_imputed.to_csv("combined_imputed_4_visits.csv", index=False)
print("Saved!", df_imputed.shape)
df_imputed.head(12)